# ChemBreak Task Bank Generator V5

This notebook does one job: generate a controlled chemistry task bank and compare four model families.

- `test`: 1 assignment, 4 real candidates, 16 blind judgments
- `pilot`: 9 assignments, 36 real candidates, 144 blind judgments

Every model generates and every model judges. Only one model is loaded on the GPU at a time.

## 1. Get the current ChemBreak code

This safely clones the repository on the first run and fast-forwards it on later runs.

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/ChemBreak")
REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository. Rename it and rerun this cell.")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

PROJECT_DIR = REPO_DIR / "ChemBreak_V5_CLEAN_PACKAGE"
PIPELINE = PROJECT_DIR / "chembreak_pipeline.py"

# Fallback: find the pipeline automatically if the V5 folder is renamed later.
if not PIPELINE.is_file():
    matches = sorted(REPO_DIR.glob("*/chembreak_pipeline.py"))
    if len(matches) == 1:
        PIPELINE = matches[0]
        PROJECT_DIR = PIPELINE.parent
    elif not matches:
        raise FileNotFoundError("No chembreak_pipeline.py file was found in any top-level repository folder.")
    else:
        raise RuntimeError(f"More than one V5 pipeline was found: {matches}")
print(f"Ready: {PROJECT_DIR}")

## 2. Install the small runtime dependency set

The notebook keeps Colab's installed PyTorch and adds only the packages needed for the four text models.

In [ ]:
import sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(PROJECT_DIR / "requirements_colab.txt"),
], check=True)

## 3. Check the GPU and Hugging Face access

Gemma is gated. Accept its license at https://huggingface.co/google/gemma-2-9b-it before continuing. If this Colab runtime has no saved Hugging Face token, a login box will appear.

In [ ]:
import os
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from huggingface_hub import get_token, notebook_login

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Choose Runtime > Change runtime type > GPU, then rerun from the top.")
print("GPU:", torch.cuda.get_device_name(0))

if get_token() is None:
    notebook_login()
else:
    print("Hugging Face login found.")

## 4. Choose the run

This is the only setting you need to change. Start with `test`. After it completes, change it to `pilot` and rerun from this cell.

In [ ]:
RUN_TYPE = "test"  # Change to "pilot" for the nine-category run.

if RUN_TYPE not in {"test", "pilot"}:
    raise ValueError('RUN_TYPE must be "test" or "pilot".')
OUTPUT_DIR = Path("/content/chembreak_v5_results") / RUN_TYPE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run type: {RUN_TYPE}")
print(f"Results:  {OUTPUT_DIR}")

## 5. Preview the controlled assignments

No model is loaded in this step. It shows exactly what all four generators will receive.

In [ ]:
import json
import pandas as pd
from IPython.display import display

subprocess.run([
    "python", str(PIPELINE),
    "--project-dir", str(PROJECT_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--run-type", RUN_TYPE,
    "--stage", "plan",
], check=True)

assignments = pd.read_csv(OUTPUT_DIR / "assignments.csv")
manifest = json.loads((OUTPUT_DIR / "run_manifest.json").read_text())
display(assignments[["plan_id", "hc_id", "hd_id", "ot_id", "chemical_entity", "context_constraint"]])
print(f"Expected candidates: {manifest['candidate_count_expected']}")
print(f"Expected judgments:  {manifest['judgment_count_expected']}")

## 6. Generate, judge, and compare

This is the main run. It uses the four real checkpoints. Rerunning the cell resumes completed candidate and judgment rows.

In [ ]:
import sys

command = [
    sys.executable, "-u", str(PIPELINE),
    "--project-dir", str(PROJECT_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--run-type", RUN_TYPE,
    "--stage", "all",
]

print("Starting ChemBreak. Progress will appear below.\n", flush=True)
with subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)

if process.returncode != 0:
    raise subprocess.CalledProcessError(process.returncode, command)
print("\nChemBreak run completed.", flush=True)

## 7. Inspect the result

The provisional bank contains at most one consensus winner for each controlled assignment. The two comparison tables show how the generators and judges differed.

In [ ]:
summary = json.loads((OUTPUT_DIR / "run_summary.json").read_text())
print(json.dumps(summary, indent=2))

print("\nGenerator comparison")
display(pd.read_csv(OUTPUT_DIR / "generator_comparison.csv"))

print("\nJudge comparison")
display(pd.read_csv(OUTPUT_DIR / "judge_comparison.csv"))

print("\nProvisional task bank")
bank = pd.read_csv(OUTPUT_DIR / "provisional_task_bank.csv")
display(bank[["candidate_id", "hc_id", "chemical_entity", "benchmark_prompt", "mean_cross_family_score"]])

## 8. Download this run

The archive is built from the same `OUTPUT_DIR` used above. The required files are checked before the ZIP is created.

In [ ]:
import shutil
import zipfile
from google.colab import files as colab_files

required = [
    "candidate_tasks_multimodel.csv",
    "judgments_multimodel.csv",
    "candidate_consensus.csv",
    "provisional_task_bank.csv",
    "run_summary.json",
]
missing = [name for name in required if not (OUTPUT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Run step 6 first. Missing result files: {missing}")

archive_base = Path("/content") / f"ChemBreak_V5_{RUN_TYPE}_results"
zip_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=str(OUTPUT_DIR)))
with zipfile.ZipFile(zip_path) as archive:
    bad_file = archive.testzip()
    if bad_file is not None:
        raise RuntimeError(f"ZIP verification failed at {bad_file}")
print(f"Archive ready: {zip_path} ({zip_path.stat().st_size:,} bytes)")
colab_files.download(str(zip_path))